# Assignment 5: Neural Network Quantization

In the previous assignments, you trained and inspected image-based driving policies. This assignment asks a related deployment question:

**What would have to change if a trained CNN had to run on a device with a limited model-size and latency budget?**

This assignment does not rely on previous assignments. You will use a provided plain CIFAR-10 CNN from `cifar10_cnn.py` and a pretrained checkpoint in `checkpoints/cifar10_cnn.pt`.

## Learning Goals

By the end of this assignment, you should be able to:

- inspect a PyTorch model's layers, parameters, activations, and output shapes,
- measure accuracy, CPU latency, model size, and approximate FLOPs,
- apply post-training static quantization with representative calibration data,
- study how calibration size affects quantized-model quality,
- choose a model variant for a deployment budget.

## Setup

The reference checkpoint is available at:

```text
checkpoints/cifar10_cnn.pt
```

In [37]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.profiler import profile, ProfilerActivity
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from torchvision.datasets import CIFAR10
from cifar10_cnn import CifarCNN, cifar10_train_transform, cifar10_eval_transform, load_reference_model
import random
import time

torch.manual_seed(42)
generator = np.random.seed(42)

print("PyTorch version:", torch.__version__)
print("Quantized engines:", torch.backends.quantized.supported_engines)

PyTorch version: 2.8.0+cu126
Quantized engines: ['qnnpack', 'none', 'onednn', 'x86', 'fbgemm']


## Task 1: Inspect The Reference Model

Start by loading the CIFAR-10 dataset and the reference CNN. Before changing precision or quantizing anything, inspect the model and what it computes. This gives you a baseline for later tasks, where you will compare modified versions of the same model.

CIFAR-10 is a standard image classification dataset containing small `32×32` color images from 10 classes.

Create a `DataLoader` from the CIFAR-10 test split for evaluation and a calibration `DataLoader` from a small representative subset of the CIFAR-10 training data. The calibration loader is used later to estimate activation ranges for quantization. It is not used for training and does not require labels.

### What you need to do

1. Create a `DataLoader`from the CIFAR-10
    - training split that will be used as the calibration data later on.
    - test split that will be used to evaluat models.
2. Load the provided reference model.
3. Print the model architecture.
4. Print one input batch shape and one output batch shape.
5. Count model parameters.
6. Register forward hooks or use another method to print all intermediate activation shapes in the format:

    ```python
    {'name': ..., 'module': ..., 'shape': ..., 'dtype': ..., 'quantized': ...}
    ```

*Notes:*
- Use the data transforms and model-loading information provided in `cifar10_cnn.py`.
- The calibration data should come from the CIFAR-10 training split, not the test split.
- Calibration labels are not used, but the `DataLoader` may still return `(image, label)` pairs.
- When printing activation shapes, record leaf modules only. Otherwise the trace may contain duplicated or uninformative entries.
- Your activation trace should include each tensor’s shape, dtype, and whether it is quantized.

**Task Output:** Model summary, parameter counts, input/output shapes, and activation trace.

In [35]:
DEVICE = torch.device("cpu")
print("Deployment device:", DEVICE)

DATA_DIR = Path("data/")

CHECKPOINT_PATH = Path("checkpoints/cifar10_cnn.pt")
print("Checkpoint exists:", CHECKPOINT_PATH.exists())

BATCH_SIZE = 128
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# TODO create the CIFAR-10 test dataloader
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=False, transform=cifar10_eval_transform())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# TODO create the calibration dataloader from a small subset of the CIFAR-10 training data
full_train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=cifar10_train_transform)

calibration_size = 512
calibration_indices = random.sample(range(len(full_train_dataset)), calibration_size)
calibration_dataset = Subset(full_train_dataset, calibration_indices)
calibration_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# TODO load the pre-trained model checkpoint and prepare it for inference
model = load_reference_model(CHECKPOINT_PATH, device=DEVICE)
model.eval()

# TODO print the input and output shape for a single batch of test data
input_batch, _ = next(iter(test_loader))
input_batch = input_batch.to(DEVICE)

with torch.no_grad():
    output_batch = model(input_batch)

print("Input batch shape:", input_batch.shape)
print("Output batch shape:", output_batch.shape)

# TODO print the number of parameters in the model
num_params = sum(p.numel() for p in model.parameters())
print("Number of parameters:", num_params)

# TODO register forward hooks to print the shape of all intermediate activations during a forward pass
activation_trace = []
hooks = []

def make_hook(name):
    def hook_fn(module, inputs, output):
        if len(list(module.children())) > 0:
            return
        
        if isinstance(output, torch.Tensor):
            entry = {
                "name": name,
                "module": module.__class__.__name__,
                "shape": tuple(output.shape),
                "dtype": str(output.dtype),
                "quantized": output.is_quantized,
            }
        elif isinstance(output, (tuple, list)) and len(output) > 0 and isinstance():
            out = output[0]
            entry = {
                "name": name,
                "module": module.__class__.__name__,
                "shape": tuple(out.shape),
                "dtype": str(out.dtype),
                "quantized": out.is_quantized,
            }

        else:
            entry = {
                "name": name,
                "module": module.__class__.__name__,
                "shape": None,
                "dtype": None,
                "quantized": None,
            }

        activation_trace.append(entry)

    return hook_fn

for name, module in model.named_modules():
    if len(list(module.children())) == 0:
        hooks.append(module.register_forward_hook(make_hook(name)))

with torch.no_grad():
    _ = model(input_batch)

for hook in hooks:
    hook.remove()

    for entry in activation_trace:
        print(entry)

Deployment device: cpu
Checkpoint exists: True
Input batch shape: torch.Size([128, 3, 32, 32])
Output batch shape: torch.Size([128, 10])
Number of parameters: 814570
{'name': 'features.conv1', 'module': 'Conv2d', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.bn1', 'module': 'BatchNorm2d', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.relu1', 'module': 'ReLU', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.conv2', 'module': 'Conv2d', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.bn2', 'module': 'BatchNorm2d', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.relu2', 'module': 'ReLU', 'shape': (128, 32, 32, 32), 'dtype': 'torch.float32', 'quantized': False}
{'name': 'features.pool1', 'module': 'MaxPool2d', 'shape': (128, 32, 16, 16), 'dtype': 'torch.float32', 'q

## Task 2: Build Deployment Metrics

Imagine a device with limited resources. To run your model on this device, you want it to satisfy a deployment budget:

* model size below `SIZE_BUDGET_MB`,
* latency below `LATENCY_BUDGET_MS_PER_IMAGE`,
* accuracy above `MIN_ACCURACY`.

Before optimizing the model, you first need reliable metrics for comparing different versions.

### Background: Memory, latency, and FLOPs

A model uses memory mainly for its parameters, such as convolution and linear layer weights. During inference, it also uses memory for intermediate activations, which are the tensors produced between layers. Runtime memory can also include temporary buffers, framework overhead, and workspace memory used by some operations.

The serialized model size usually measures how much space the saved model takes on disk. This is often close to the size of the saved weights, but it can also include metadata, buffers, or compression depending on the file format. Runtime parameter memory also depends on the numerical precision used, such as FP32, FP16, or INT8.

Latency measures how long inference takes. In this task, report average latency per image. Use the same device, batch size, preprocessing, and measurement method for all models so that comparisons are fair. When timing on a GPU, use warmup runs and synchronization so that the measured time reflects actual inference time.

FLOPs means floating-point operations. It estimates how many arithmetic operations, such as multiplications and additions, are needed for one forward pass. FLOPs describe the amount of computation, not how fast the hardware performs it.

This should not be confused with FLOPS, or FLOP/s, which means floating-point operations per second. FLOP/s is a speed rate. For example, FLOPs describe the work required by the model, while FLOP/s describes how quickly a device can perform that work.

Different libraries may report FLOPs differently. In this assignment, we first count MACs, where one MAC means one multiply-add. If we count the multiply and the addition as separate floating-point operations, then one MAC is approximately 2 FLOPs. Some tools instead report one MAC as one operation, so FLOP counts from different tools may not match exactly unless they use the same convention.

### Useful formulas

For classification accuracy:

$$
\text{Accuracy} = \frac{\text{\# correct predictions}}{\text{\# examples}}
$$

For average latency per image:

$$
\text{Latency per image} = \frac{\text{total inference time}}{\text{\# images}}
$$

For serialized model size:

$$
\text{Model size in MiB} = \frac{\text{saved model size in bytes}}{1024^2}
$$

For an approximate Conv2d layer cost, count multiply-accumulate operations, or MACs, first:

$$
\text{MACs} =
H_{out} \cdot W_{out} \cdot C_{out} \cdot
\frac{C_{in}}{\text{groups}} \cdot
K_h \cdot K_w
$$

where, $H_{out}$ and $W_{out}$ are the output feature map height and width, $C_{out}$ is the number of output channels, $C_{in}$ is the number of input channels, `groups` is the number of channel groups in the convolution, and $K_h$ and $K_w$ are the kernel height and width.

If one multiply-add is counted as 2 floating-point operations:

$$
\text{FLOPs} \approx 2 \cdot \text{MACs}
$$

This approximation counts the main convolution computation. It usually ignores smaller costs such as bias additions, activation functions, padding, pooling, batch normalization, and other framework-specific overhead.

For a Linear layer:

$$
\text{MACs} = \text{input features} \cdot \text{output features}
$$

Again, if one multiply-add is counted as 2 floating-point operations:

$$
\text{FLOPs} \approx 2 \cdot \text{MACs}
$$

FLOPs describe how much computation one forward pass requires. FLOP/s describes device speed:

$$
\text{FLOP/s} = \frac{\text{number of floating-point operations}}{\text{time in seconds}}
$$


### What you need to do

**Implement reusable functions** for calculating:

1. classification accuracy,
2. average inference time per image in milliseconds,
3. serialized model size in MB,
4. approximate FLOPs for one forward pass,
5. estimate FLOPs through the `torch.profiler` API
6. a result table that marks whether a model satisfies the deployment budget.

Use your model loaded in Task 1 to compute and ensure it is in `float32` precision.

*Notes:*
- Use `model.eval()` for all metrics.
- Disable gradients during evaluation and timing.
- Use the same batch size for all models.
- Run a few warm-up batches before measuring latency.
- Time only inference, not data loading.
- Report latency as average **milliseconds per image**.
- For FLOPs, count each `Conv2d` and `Linear` layer once.
- Measure serialized size by saving the model `state_dict`, not by saving the entire Python model object.

### Suggested table

| Model              | Accuracy | Latency ms/image | Size MB | our FLOPs | profiler FLOPs | Budget pass/fail |
| ------------------ | -------- | ---------------- | ------- | --------- | -------------- | ---------------- |
| `float32` baseline |          |                  |         |           |                |                  |

### Task output

**Task Output:** Metric functions and a printed `float32` baseline table with one row.

In [39]:
LATENCY_BUDGET_MS_PER_IMAGE = 0.20
SIZE_BUDGET_MB = 1.25
MIN_ACCURACY = 0.80

def compute_accuracy(model, dataloader, device = DEVICE):
    model.eval()
    correct = 0
    total = 0

    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

def compute_latency_ms_per_image(model, dataloader, device = DEVICE, warmup_batches=5, num_batches=10):
    model.eval()

    total_time = 0.0
    total_images = 0

    with torch.inference_mode():
        iterator = iter(dataloader)

        for _ in range(warmup_batches):
            try:
                inputs, _ = next(iterator)
            except StopIteration:
                return float("nan")
            inputs = inputs.to(device)
            _ = model(inputs)

        for _ in range(num_batches):
            try:   
                inputs, _ = next(iterator)
            except StopIteration:
                return float("nan")
            inputs = inputs.to(device)
            
            
            start = time.perf_counter()
            _ = model(inputs)
            end = time.perf_counter()

            total_time += end - start
            total_images += inputs.size(0)

    if total_images == 0:
        return float("nan")
    
    return (total_time/total_images) * 1000.0

def compute_model_size_mib(model, path="tmp_model.pt"):
    path = Path(path)
    torch.save(model.state_dict(), path)
    size_bytes = path.stat().st_size
    path.unlink()
    return size_bytes / (1024**2)

def compute_model_flops_ours():
    macs = (
        32 * 32 * 32 * 3 * 3 * 3 + # conv1
        32 * 32 * 32 * 32 * 3 * 3 + # conv2
        16 * 16 * 64 * 32 * 3 * 3 + # conv3
        16 * 16 * 64 * 64 * 3 * 3 + # conv4
        8 * 8 * 128 * 64 * 3 * 3 + # conv5
        8 * 8 * 128 * 128 * 3 * 3 + # conv6
        2048 * 256 + # fc1
        256 * 10 # fc2
    )

    return 2 * macs


def compute_flops_profiler(model, input_batch, device=DEVICE):
    model.eval()
    input_batch = input_batch.to(device)

    with profile(
        activities=[ProfilerActivity.CPU],
        with_flops=True,
        record_shapes=True,
    ) as prof:
        with torch.inference_mode():
            _= model(input_batch)

    total_flops = 0
    for item in prof.key_averages():
        if item.flops is not None:
            total_flops += item.flops

    return total_flops

def passes_budget(accuracy, latency_ms_per_image, size_mib):
    return (
        accuracy >= MIN_ACCURACY
        and latency_per_image_ms <= LATENCY_BUDGET_MS_PER_IMAGE
        and size_mib <= SIZE_BUDGET_MB
    )






# TODO set model to evaluation mode
model.eval()

# TODO compute model accuracy on the test set
accuracy = compute_accuracy(model, test_loader, device=DEVICE)

# TODO compute average latency per image on the test set
latency_per_image_ms = compute_latency_ms_per_image(model, test_loader, device=DEVICE)

# TODO compute serialized model size in MiB
model_size_mib = compute_model_size_mib(model)

# TODO compute and print the approximate number of FLOPs for a single forward pass through the model
# Note: compute FLOPs by hand using the formulas provided in the notebook and compare it to the FLOPs computed by `torch.profiler` for verification.
input_batch, _ = next(iter(test_loader))
model_flops_ours = compute_model_flops_ours()
model_flops_profiler = compute_flops_profiler(model, input_batch, device=DEVICE)

# TODO print the results and check if they meet the specified constraints
budget_ok = passes_budget(accuracy, latency_per_image_ms, model_size_mib)

print(
    f"| float32 baseline | {accuracy:.4f} | {latency_per_image_ms:.4f} | "
    f"{model_size_mib:.4f} | {model_flops_ours:,} | "
    f"{model_flops_profiler:,} | {'PASS' if budget_ok else 'FAIL'} | "
)


| float32 baseline | 0.8147 | 0.6455 | 3.1236 | 78,320,640 | 10,025,041,920 | FAIL | 


## Task 3: Precision Experiments

Machine learning models can be stored and executed using different numeric precisions. The default is usually `float32`, but lower-precision formats such as `float16` and `bfloat16` can reduce storage and memory bandwidth. They may also change accuracy, latency, or hardware compatibility.

<div style="text-align: center;">
<img src="https://images.contentstack.io/v3/assets/blt71da4c740e00faaa/blt40c8ab571893763a/65f370cc0c744dfa367c0793/EXX-blog-fp64-fp32-fp-16-5_(3).jpg" width="800" style="center"/>
</div>

In this task, compare precision variants on CPU. Some variants may fail or run slower depending on the PyTorch build and CPU hardware. That is part of the experiment.

### What you need to do

1. Use your `float32` model as the baseline.
2. Create manual `float16` and `bfloat16` model copies by converting both the model and input tensors to the same dtype.
3. Evaluate each version using the same data and metric functions.
4. [If applicable] Catch and report unsupported-operation errors instead of letting the notebook stop.
5. Evaluate CPU `float16` autocast as a separate variant. When you use autocast, keep the model and inputs in `float32` and wrap only the forward pass in:

    ```python
    with torch.autocast(device_type="cpu", dtype=torch.float16):
        outputs = model(inputs)
    ```
6. Record:
   - accuracy,
   - average latency in ms/image,
   - serialized model size,
   - whether the model ran successfully,
   - any warnings, errors, or hardware limitations.

### Suggested table

| Version | Accuracy | Latency ms/image | Size MB | Runs? | Notes |
|---|---:|---:|---:|---|---|
| `float32` |  |  |  | Yes | Baseline |
| manual `float16` |  |  |  |  |  |
| manual `bfloat16` |  |  |  |  |  |
| CPU autocast `float16` |  |  |  |  | Keep model/input in FP32 |

**Task output:** Submit a precision comparison table and a short interpretation. Explain which version was fastest or smallest, whether accuracy changed, and whether the result would be practical for CPU deployment.

In [ ]:
# TODO initialize a float16 model copy

# TODO initialize a bfloat16 model copy

# TODO reuse your functions from Task 2 for calculating metrics

# TODO test your float32 model version with float16 autocasting

## Task 4: CPU INT8 Post-Training Static Quantization

In this task, you will apply **post-training static quantization** to your trained CNN and compare it against the original `float32` model.

Modern deep learning models are usually trained in floating point precision, commonly `float16`, `bloat16`, or `float32`. This is convenient because it gives the model enough numerical precision during learning. However, once a model has already been trained, we often care about different things: model size, memory usage, inference latency, and whether the model can run efficiently on limited hardware, e.g., on CPUs.

Quantization is one way to address this. Instead of storing and computing with (full-precision) floating-point values, quantization represents some values using lower-precision formats, commonly **8-bit integers**. This can reduce model size and memory bandwidth and may improve inference speed, especially on CPUs, which often have highly optimized integer arithmetic. The trade-off is that lower precision gives the model fewer distinct numerical values to work with, which can reduce accuracy.

### Background

In our **post-training static quantization**, we start with a trained `float32` model. Before converting it to an integer representation, we run some representative input data through the model. This step is called **calibration**.

During calibration, PyTorch observes the ranges of activations inside the model. These observed ranges are then used to choose quantization parameters such as **scales** and **zero-points**. These parameters determine how floating-point values are mapped into integer buckets.

<div style="text-align: center;"><img src="https://substackcdn.com/image/fetch/$s_!yYxw!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fe9d17077-d9af-4b37-9b9b-57ef9aaa1ca9_680x486.png" width="400"/></div>

Another important step is **module fusion**. Certain sequences of layers, such as `Conv2d-BatchNorm2d-ReLU`, can be mathematically combined into a single equivalent operation before quantization. Fusion reduces the number of separate computations and intermediate activations, which can improve inference efficiency and often leads to better quantization results because fewer quantization and dequantization boundaries are needed.

For this task, you will use PyTorch’s classic eager-mode quantization workflow. This workflow is useful for learning because it makes the main steps visible:

* adding quantization and dequantization stubs,
* fusing compatible layers,
* attaching a quantization configuration,
* calibrating the model,
* converting the model to quantized form,
* and inspecting the resulting quantized modules.

In current PyTorch development, newer quantization workflows are increasingly moving toward `torchao` and PT2E-style APIs. However, for this course task, the classic workflow is useful because it exposes the mechanics clearly and works well for our small CPU-based experiment.

### What you need to do

You should compare your original `float32` CNN against several statically quantized `int8` versions of the same model.

Use CPU only for the quantized model.

1. Evaluate the original `float32` model.

   * Measure accuracy.
   * Measure average inference latency.
   * Measure serialized model size.

2. Define a small quantization wrapper around your CNN.

   * Add a `QuantStub` before the model.
   * Add a `DeQuantStub` after the model.

3. For each calibration setting, create a fresh copy of the trained `float32` model.

   * Use calibration sizes such as 1, 5, and 30 mini-batches.
   * Do not reuse an already-prepared or already-converted model.

4. Fuse compatible modules where appropriate.

   * Examples of valid patterns include:

     * `Conv2d-BatchNorm2d-ReLU`
     * `Conv2d-ReLU`
     * `Linear-ReLU`
   * Only fuse modules that are actually adjacent in your model.

5. Attach a backend-specific quantization configuration.

   * Select a quantized backend supported by your machine.
   * Use a matching default `qconfig`.

6. Prepare the model for static quantization.

7. Calibrate the model.

   * Use representative training data from the previously implemented calibration dataloader.
   * Use the same preprocessing as during evaluation.
   * Do not use the test set for calibration.

8. Convert the calibrated model to an `int8` quantized model.

9. Evaluate each quantized model.

   * Measure accuracy.
   * Measure average inference latency.
   * Measure serialized model size.
   * Compare against the original `float32` baseline.

10. Inspect one quantized model.
    Print  information such as:

    * the model structure before and after fusion,
    * the model structure after quantization,
    * selected layer types,
    * selected `state_dict` entries,
    * tensor dtypes,
    * quantization scales,
    * zero-points.

### Suggested comparison table

| Model         | Calibration batches | Accuracy | Accuracy drop | Avg latency | Speedup vs FP32 | Model size | Size reduction | Notes |
| ------------- | ------------------- | -------- | ------------- | ----------- | --------------- | ---------- | -------------- | ----- |
| FP32 baseline | —                   |          | —             |             | 1.00×           |            | 1.00×          |       |
| INT8 static   | 1                   |          |               |             |                 |            |                |       |
| INT8 static   | 5                   |          |               |             |                 |            |                |       |
| INT8 static   | 30                  |          |               |             |                 |            |                |       |

### Important notes

* Call `model.eval()` before fusion, preparation, calibration, conversion, and evaluation.
* Use `torch.inference_mode()` or `torch.no_grad()` during calibration and evaluation.
* Use warm-up iterations before measuring latency.
* Measure latency over multiple batches or repeated runs.
* Keep the batch size fixed when comparing models.
* When measuring model size, compare serialized model files in a consistent way.
* When inspecting the quantized model, do not assume every `state_dict` entry is a floating-point tensor. Some entries may be quantized tensors, integer tensors, scales, zero-points, or packed parameters.

**Task output:**

* your quantization wrapper,
* your fusion, preparation, calibration, and conversion code,
* the baseline and quantized comparison table,
* and selected inspection output from one quantized model.

In [ ]:
# TODO Your quantization implementation

## Task 5: Final Comparison And Deployment Decision

Make a final comparison table or figure from your results.

### What you need to do

1. Compare all evaluated variants:
   - `float32` baseline,
   - precision variants,
   - quantized models for each calibration setting.
2. Show accuracy, latency, serialized model size, and budget pass/fail.
3. Clearly mark which variants satisfy all deployment constraints.
4. Choose the model you would deploy.
5. Justify your choice using the measured trade-off between accuracy, latency, and model size.

**Task Output:** Final comparison and deployment recommendation.

In [ ]:
# TODO gather your final metrics from previous tasks here

**Your Answer:**

\<Your Answer Here\>